# Generation + Formatting
- Tool : Tavily

In [ ]:
# !pip install langchain-tavily -qqq

In [ ]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(max_results=1)

### Generator

In [ ]:
from typing import List
from typing_extensions import TypedDict

class GraphState(TypedDict):
    user_input: str         # 사용자 질문
    documents: List[str]    # 검색된 문서
    generation: str         # LLM 생성 결과
    source_urls: str        # tavily 생성 결과    

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 프롬프트 문구 작성
system_generate = '''
당신은 사용자의 질문에 대해 주어진 context에 기반해 답변하는 부동산 전문가입니다.
주어진 context는 vectorstore에서 검색된 결과입니다. 이를 기반으로 아래 내용을 참고하여 사용자의 user_input에 답변하세요.
사용자가 링크를 요청하면 'tavily' 도구를 호출하여 URL을 확보한 뒤 답변에 포함하세요.

[역할(Role)]
- 주택임대차보호법, 판례, 법령해석례를 기반으로 사용자가 계약 관련 궁금증을 해소할 수 있도록 정확한 정보를 안내합니다.
- 복잡한 법률 용어를 일반인도 이해할 수 있도록 쉽게 설명합니다.
- 제공된 법령/판례/해석례 문서만을 근거로 답변합니다.

[제약조건(Constraints)]
- 제공된 context에 없는 내용은 절대 생성하거나 추론하지 마세요.
- 모든 답변에 반드시 출처(법령명, 조문번호, 판례번호)를 명시하세요.
- 할루시네이션 방지를 위해 문서에 명시된 내용만 인용합니다.
- 법률적 판단이나 개인 법률 자문은 제공하지 않습니다.
==========================
user_input: {user_input}
context: {context}
'''

# prompt 생성
prompt_generate = PromptTemplate(
    input_variables=['user_input', 'context'],
    template=system_generate
)

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")
tools = [tavily_tool]
model_with_tools = model.bind_tools(tools)   # 도구 바인딩

output_parser = StrOutputParser()

# Chain 생성
chain_generate = prompt_generate | model_with_tools | output_parser

In [ ]:
def generate(state):
    '''
    검색된 문서(데이터)를 활용해 사용자 질문에 대한 답변을 생성합니다.
    
    Args:
        state (dict): 현재 graph state
    Return:
        state (dict): LLM 생성 결과와 사용자 질문을 포함하는 새로운 graph state
    '''

    user_input = state['user_input']
    documents = state['documents']
    response = chain_generate.invoke({"user_input": user_input, "context": documents})

    return {
        'user_input': user_input,
        'documents': documents,
        'generation': response
    }

### Formatter

In [ ]:
# 프롬프트 문구 작성
system_formatting = '''
당신은 Generation에 대해 답변의 스타일을 정제해 주는 AI 어시스턴트입니다.
주어진 Generation은 Generator에 의해 생성된 답변입니다. 아래 Answer Style을 기반으로 Generation을 정제하세요.

** Answer Style **
[출력 형식 (Format)]
답변: (핵심 내용을 쉬운 말로 설명하되 확답의 표현은 자제)

관련 법령/판례:
  - [출처] [법령명 또는 판례번호] : (법령/판례의 핵심 내용)

* 본 내용은 법률 자문이 아닌 법령 정보 안내이며,
  구체적인 법적 판단은 전문가와 상담하시기 바랍니다.
==========================
generation: {generation}
'''

# prompt 생성
prompt_formatting = PromptTemplate(
    input_variables=['generation'],
    template=system_formatting
)

output_parser = StrOutputParser()

# Chain 생성
chain_formatting = prompt_formatting | model | output_parser


In [ ]:
def formatting(state):
    '''
    생성된 답변을 Answer Style에 맞게 정제합니다.
    
    Args:
        state (dict): 현재 graph state
    Return:
        state (dict): 답변 정제 결과를 포함하는 새로운 graph state
    '''

    user_input = state['user_input']
    documents = state['documents']
    generation_gen = state['generation']    # 위 Generator가 생성한 답변
    generation_formatting = chain_formatting.invoke({"generation": generation_gen})

    return {
        'user_input': user_input,
        'documents': documents,
        'generation': generation_formatting.content
    }

### Tavily tool

In [ ]:
@tool
def tavily(query: str) -> str:
    '''
    사용자가 법령, 판례의 원문 링크나 상세 주소(URL)를 요청할 때만 이 도구를 사용하세요.
    입력값(query)은 검색에 최적화된 검색어여야 합니다.

    Args:
        query(str) : 찾고자 하는 것
    Return:
        url(str) : query에 적합한 법령, 판례 링크
    '''

    result_tavily = tavily_tool.invoke(query)
    url = result_tavily['result'][0]['url']

    return url

### LangGraph 그리기

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(GraphState)

tool_node = ToolNode(tools)

workflow.add_node('generate', generate)
workflow.add_node('tools', tool_node)
workflow.add_node('formatting', formatting)

workflow.add_edge(START, 'generate')
workflow.add_conditional_edges('chatbot', tools_condition, {'tools':'tools', 'formatting':'formatting'})
workflow.add_edge('tools', 'generate')
workflow.add_edge('formatting', END)

graph = workflow.compile()
graph

##### Graph use

In [ ]:
user_query = 'LangGraph가 뭐죠? 모르면 모른다고 답변하세요.'

state = {'messages' : [HumanMessage(content=user_query)]}

response = graph.invoke(state)
response['messages'][-1].content